<a href="https://colab.research.google.com/github/tomonari-masada/course2026-sml/blob/main/08_logistic_regression.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# ロジスティック回帰による糖尿病の予測

* 有名なPima Indians Diabetes Databaseを使う（下リンク先）

  * https://www.kaggle.com/uciml/pima-indians-diabetes-database

* ロジスティック回帰、そして、分類の評価については、下記も参照
  * https://developers.google.com/machine-learning/crash-course/logistic-regression/
  * https://developers.google.com/machine-learning/crash-course/classification/

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split
from sklearn.impute import SimpleImputer, KNNImputer
from sklearn.linear_model import LogisticRegression, LinearRegression
from sklearn.preprocessing import MinMaxScaler, StandardScaler
from sklearn.metrics import (
    roc_auc_score, RocCurveDisplay,
    average_precision_score, PrecisionRecallDisplay
)
from sklearn.model_selection import StratifiedKFold

%config InlineBackend.figure_format = 'retina'

## データの読み込み

In [ ]:
diabetes = pd.read_csv('/content/drive/MyDrive/data/diabetes.csv')

In [ ]:
diabetes.head()

In [ ]:
y = diabetes['Outcome']
X = diabetes.drop('Outcome', axis=1)

## 訓練データ、テストデータに分割

**この分割は変えないようにしてください。**

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.15, random_state=123)

In [ ]:
X_train.describe()

In [ ]:
X_train.hist(bins=50, figsize=(12,12));

## ベースライン: デフォルト設定のロジスティック回帰
* 交差検証も何もせずに、単にテストセット以外の部分で、モデルの学習を実行する。

In [ ]:
baseline = LogisticRegression(random_state=123)
baseline.fit(X_train, y_train)

* `max_iter`が小さいとの警告が出ているので、増やして学習しなおし。

In [ ]:
baseline = LogisticRegression(max_iter=1000, random_state=123)
baseline.fit(X_train, y_train)

* 大丈夫だったので、テストデータでの最終評価値を得る。
  * scoreメソッドを使う。

In [ ]:
print(f'test score: {baseline.score(X_test, y_test):.4f}')

* Area under ROC curveも計算してみる。


In [ ]:
y_test_pred_proba = baseline.predict_proba(X_test)
print(f'ROC AUC: {roc_auc_score(y_test, y_test_pred_proba[:,1]):.4f}')

* ROC curveを描いてみる。
  * https://scikit-learn.org/stable/auto_examples/model_selection/plot_roc.html#sphx-glr-auto-examples-model-selection-plot-roc-py

In [ ]:
display = RocCurveDisplay.from_estimator(baseline, X_test, y_test, name="baseline")
display.ax_.set_title("ROC curve");

* precision-recall curveを描いてみる。

In [ ]:
display = PrecisionRecallDisplay.from_estimator(baseline, X_test, y_test, name="baseline")
display.ax_.set_title("precision-recall curve");

In [ ]:
average_precision = average_precision_score(y_test, y_test_pred_proba[:,1])
print(f'average precision: {average_precision:.4f}')

* これをベースラインとみなす。
  * これより良い結果を得るべく、試行錯誤する。
* 試行錯誤した結果として辿り着いたモデルで、**最後に一回、テストデータ上での評価**を行う。

---

**以下、訓練データ部分を使って、交差検証によって良いモデルを探す。**

* ここに示すのは一つの試行錯誤の例なので、これに従わなくても全然大丈夫です。
---



## 交差検証しつつ試行錯誤

### 交差検証の準備

* `StratifiedKFold`クラスを使う。

In [ ]:
skf = StratifiedKFold(n_splits=10, shuffle=True, random_state=123)

* 交差検証のためのヘルパ関数

In [ ]:
def cv(skf, X_train, y_train, preprocess=None, max_iter=1000, verbose=True, **kwargs):

  # キーワード引数として、モデルの設定を指定できるようにしてある。
  if verbose:
    for key, value in kwargs.items():
      print(f'{key} = {value}')

  # 交差検証のループ
  scores = []
  for train_index, valid_index in skf.split(X_train, y_train):

    cv_X_train = X_train.iloc[train_index]
    cv_y_train = y_train.iloc[train_index]
    cv_X_valid = X_train.iloc[valid_index]
    cv_y_valid = y_train.iloc[valid_index]

    # データの前処理
    #   その都度、関数preprocessを定義してから、この関数cvを呼び出す。
    if preprocess:
      cv_X_train, cv_X_valid = preprocess(cv_X_train, cv_X_valid, verbose=verbose)

    # ロジスティック回帰の学習
    model = LogisticRegression(**kwargs, max_iter=max_iter)
    model.fit(cv_X_train, cv_y_train)

    # 検証データでの評価
    #score = model.score(cv_X_valid, cv_y_valid)
    score = average_precision_score(cv_y_valid, model.predict_proba(cv_X_valid)[:,1])
    if verbose:
      print(f'  average precision: {score:.4f}')
    scores.append(score)

  mean_score = np.mean(scores)
  print(f'mean average precision: {mean_score:.4f}')
  return mean_score

### デフォルトの設定での評価
* 交差検証で性能評価するとどうなるかを確認している。

In [ ]:
cv(skf, X_train, y_train);

### BloodPressureへの対応

* まず、属性「BloodPressure」について、ヒストグラムを描いてよくよく眺める。


In [ ]:
sns.histplot(X_train['BloodPressure']);

* 0という値がけっこうあるらしい。実は、これは欠測値。
* そこで、中央値で置き換えることにする。

In [ ]:
X_train_copy = X_train.copy()

feature = 'BloodPressure'
imp = SimpleImputer(missing_values=0, strategy='median')
X_train_copy[feature] = imp.fit_transform(X_train[[feature]])
print(f'imputation fill value for {feature}: {imp.statistics_[0]}')

sns.histplot(X_train_copy[feature]);

* 欠測箇所を中央値で埋める関数を定義しておく。
  * これは、交差検証を実行するときに使用する。

In [ ]:
def preprocess_bp(X_train, X_valid, verbose=True):
  imp = SimpleImputer(missing_values=0, strategy='median')

  X_train_copy, X_valid_copy = X_train.copy(), X_valid.copy()

  feature = 'BloodPressure'
  X_train_copy[feature] = imp.fit_transform(X_train[[feature]])
  X_valid_copy[feature] = imp.transform(X_valid[[feature]])
  if verbose:
    print(f'  imputation fill value for {feature}: {imp.statistics_[0]}')

  return X_train_copy, X_valid_copy

* 交差検証で評価する。

In [ ]:
cv(skf, X_train, y_train, preprocess=preprocess_bp);

### BMIへの対応

* 次に、training dataの「BMI」のヒストグラムを描いてみる


In [ ]:
sns.histplot(X_train['BMI']);

* やはり欠測部分が0とされているようなので、先ほどと同様、中央値で埋める。


In [ ]:
X_train_copy = X_train.copy()

feature = 'BMI'
imp = SimpleImputer(missing_values=0, strategy='median')
X_train_copy[feature] = imp.fit_transform(X_train[[feature]])
print(f'imputation fill value for {feature}: {imp.statistics_[0]}')

sns.histplot(X_train_copy[feature]);

* 交差検証で評価する。
  * 欠測箇所を埋める関数を書き換える。

In [ ]:
def preprocess_bp_bmi(X_train, X_valid, verbose=True):
  imp = SimpleImputer(missing_values=0, strategy='median')

  X_train_copy, X_valid_copy = X_train.copy(), X_valid.copy()

  for feature in ['BloodPressure', 'BMI']:
    X_train_copy[feature] = imp.fit_transform(X_train[[feature]])
    X_valid_copy[feature] = imp.transform(X_valid[[feature]])
    if verbose:
      print(f'  imputation fill value for {feature}: {imp.statistics_[0]}')

  return X_train_copy, X_valid_copy

In [ ]:
cv(skf, X_train, y_train, preprocess=preprocess_bp_bmi);

### Glucoseへの対応

In [ ]:
sns.histplot(X_train['Glucose']);

In [ ]:
X_train_copy = X_train.copy()

feature = 'Glucose'
imp = SimpleImputer(missing_values=0, strategy='median')
X_train_copy[feature] = imp.fit_transform(X_train[[feature]])
print(f'imputation fill value for {feature}: {imp.statistics_[0]}')

sns.histplot(X_train_copy[feature]);

* 欠測箇所を埋める関数を書き換える。

In [ ]:
def preprocess_bp_bmi_gl(X_train, X_valid, verbose=True):
  imp = SimpleImputer(missing_values=0, strategy='median')

  X_train_copy, X_valid_copy = X_train.copy(), X_valid.copy()

  for feature in ['BloodPressure', 'BMI', 'Glucose']:
    X_train_copy[feature] = imp.fit_transform(X_train[[feature]])
    X_valid_copy[feature] = imp.transform(X_valid[[feature]])
    if verbose:
      print(f'imputation fill value for {feature}: {imp.statistics_[0]}')

  return X_train_copy, X_valid_copy

In [ ]:
cv(skf, X_train, y_train, preprocess=preprocess_bp_bmi_gl);

**ここまででベストな結果は？**

### SkinThicknessとInsulinへの対応

In [ ]:
sns.histplot(X_train['SkinThickness'], bins=50);

In [ ]:
sns.histplot(X_train['Insulin'], bins=50);

In [ ]:
(X_train['SkinThickness'] == 0).sum().item()

In [ ]:
(X_train['Insulin'] == 0).sum().item()

* 欠測値が多すぎるので、同じ一つの値で埋めると、問題あり。

### 線形回帰で欠測箇所を埋める

* 新たに前処理の関数を定義する。
  * この関数の中で、前に使った前処理を定義した関数を呼び出すことにする。

In [ ]:
def preprocess_linreg(X_train, X_valid, verbose=True):

  # 前に使った前処理を定義した関数を呼び出す
  X_train_copy, X_valid_copy = preprocess_bp_bmi_gl(X_train, X_valid, verbose=verbose)

  # 欠測値を埋めるための回帰において特徴量として使う列
  columns = X_train_copy.columns.drop('SkinThickness').drop('Insulin')

  # 線形回帰で欠測箇所を埋める
  for feature in ['SkinThickness', 'Insulin']:
    reg = LinearRegression()
    missing_mask = X_train_copy[feature] == 0
    reg.fit(X_train_copy.loc[~missing_mask, columns], X_train_copy.loc[~missing_mask, feature])
    prediction = reg.predict(X_train_copy.loc[missing_mask, columns]).clip(min=0).astype(np.int64)
    X_train_copy.loc[missing_mask, feature] = prediction

    missing_mask = X_valid_copy[feature] == 0
    prediction = reg.predict(X_valid_copy.loc[missing_mask, columns]).clip(min=0).astype(np.int64)
    X_valid_copy.loc[missing_mask, feature] = prediction

  return X_train_copy, X_valid_copy

* 問：`clip(min=0)`を挟んであるのは、なぜか？

In [ ]:
cv(skf, X_train, y_train, preprocess=preprocess_linreg);

### k-NNを使って欠測箇所を埋める
  * ここでは`KNNImputer`を使う。（自分で実装すると、どうなる？）

In [ ]:
def make_knn_preprocess(k):
  """kを引数に取り、前処理関数を返すファクトリ関数"""
  def preprocess(X_train, X_valid, verbose=True):
    # 前に行った前処理を定義した関数を呼び出す
    X_train_copy, X_valid_copy = preprocess_bp_bmi_gl(X_train, X_valid, verbose=verbose)
    if verbose:
      print(f'imputation k-NN k={k}')
    imputer = KNNImputer(n_neighbors=k, missing_values=0)
    X_train_imputed = pd.DataFrame(
        imputer.fit_transform(X_train_copy),
        columns=X_train_copy.columns,
        index=X_train_copy.index
    )
    X_valid_imputed = pd.DataFrame(
        imputer.transform(X_valid_copy),
        columns=X_valid_copy.columns,
        index=X_valid_copy.index
    )
    return X_train_imputed, X_valid_imputed

  return preprocess

In [ ]:
best_k, best_score = 1, 0.0

for k in range(1, 21):
  score = cv(skf, X_train, y_train, preprocess=make_knn_preprocess(k))
  print('-'*64)
  if best_score < score:
    best_k, best_score = k, score

print(f'best score {best_score:.4f} for k = {best_k}')

### スケーラー

* スケーリングを行う関数の中で、上の関数を呼び出す。

In [ ]:
def make_scaler_preprocess(k, scaler_class, **scaler_kwargs):
  def preprocess(X_train, X_valid, verbose=True):
    X_train_copy, X_valid_copy = make_knn_preprocess(k)(X_train, X_valid, verbose=verbose)
    scaler = scaler_class(**scaler_kwargs)
    scaler.fit(X_train_copy)
    X_train_scaled = pd.DataFrame(
        scaler.transform(X_train_copy),
        columns=X_train_copy.columns,
        index=X_train_copy.index
    )
    X_valid_scaled = pd.DataFrame(
        scaler.transform(X_valid_copy),
        columns=X_valid_copy.columns,
        index=X_valid_copy.index
    )
    return X_train_scaled, X_valid_scaled
  return preprocess

In [ ]:
cv(skf, X_train, y_train, preprocess=make_scaler_preprocess(best_k, MinMaxScaler), verbose=False);

In [ ]:
cv(skf, X_train, y_train, preprocess=make_scaler_preprocess(best_k, StandardScaler), verbose=False);

### 正則化

In [ ]:
best_C, best_C_score = None, 0.0

for C in np.power(10.0, np.arange(10) - 4):
  score = cv(skf, X_train, y_train, preprocess=make_knn_preprocess(best_k), C=C)
  if best_C_score < score:
    best_C, best_C_score = C, score
  print('-' * 64)

print(f'best score {best_C_score:.4f} for C={best_C}')

**以上は、あくまで参考例です。**
* 例えば・・・
  * スケーリングって、欠測箇所を回帰やk-NNで埋める前に使っておかないといけないんじゃないの？

## テストデータで最終評価

* 最初と全く同じ方法でtraining set/test setの分割を用意する。

In [ ]:
diabetes = pd.read_csv('/content/drive/MyDrive/data/diabetes.csv')
y = diabetes['Outcome']
X = diabetes.drop('Outcome', axis=1)
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.15, random_state=123)

* 訓練データの中央値を使って、テストデータの欠測値を埋める。

In [ ]:
X_train_imputed = X_train.copy()
X_test_imputed = X_test.copy()
for feature in ['BloodPressure', 'BMI', 'Glucose']:
  imp = SimpleImputer(missing_values=0, strategy='median')
  X_train_imputed[feature] = imp.fit_transform(X_train[[feature]])
  X_test_imputed[feature] = imp.transform(X_test[[feature]])
  print(f'imputation fill value for {feature}: {imp.statistics_[0]}')

* k-NNでは、上で'BloodPressure', 'BMI', 'Glucose'の欠測値を埋めたデータを使う。

In [ ]:
imputer = KNNImputer(n_neighbors=best_k, missing_values=0)
X_train_imputed = pd.DataFrame(
    imputer.fit_transform(X_train_imputed),
    columns=X_train.columns, index=X_train.index
)
X_test_imputed = pd.DataFrame(
    imputer.transform(X_test_imputed),
    columns=X_test.columns, index=X_test.index
)

In [ ]:
model = LogisticRegression(max_iter=1000, C=best_C, random_state=123)
model.fit(X_train_imputed, y_train)
print('test score: {:.4f}'.format(model.score(X_test_imputed, y_test)))

In [ ]:
y_test_pred_proba = model.predict_proba(X_test_imputed)
print('ROC AUC: {:.4f}'.format(roc_auc_score(y_test, y_test_pred_proba[:,1])))

In [ ]:
fig, ax = plt.subplots(figsize=(6, 6))
RocCurveDisplay.from_estimator(baseline, X_test, y_test, name="baseline", ax=ax)
display = RocCurveDisplay.from_estimator(model, X_test_imputed, y_test, name="my model", ax=ax)
display.ax_.set_title("ROC curve");

In [ ]:
fig, ax = plt.subplots(figsize=(6, 6))
PrecisionRecallDisplay.from_estimator(baseline, X_test, y_test, name="baseline", ax=ax)
display = PrecisionRecallDisplay.from_estimator(model, X_test_imputed, y_test, name="my model", ax=ax)
display.ax_.set_title("precision-recall curve");

In [ ]:
average_precision = average_precision_score(y_test, y_test_pred_proba[:,1])
print(f'average precision: {average_precision:.4f}')




---



# 課題
* 上の結果を改良できるかどうか、試行錯誤してみてください。
  * 例：`IterativeImputer`を使うと、どうなるでしょうか？
* training setとtest setへの分割は、変更しないでください。
* training set上での試行錯誤は、どんな方法を使ってもいいです。
  * test setは、最終的な性能評価のときに一回使うだけです。